In [2]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL postgres;")
con.execute("LOAD postgres;")

con.execute("""
    CREATE SECRET pg_secret (
        TYPE postgres,
        HOST 'track1-experiment.crmc1hcwokb9.us-east-1.rds.amazonaws.com',
        PORT 5432,
        DATABASE dataset,
        USER 'user_uninorte_1',
        PASSWORD 'DFL^e!CCt'
    );
""")

con.execute("ATTACH '' AS mi_pg (TYPE postgres, SECRET pg_secret);")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [6]:
tablas = con.execute("""
    SELECT schemaname, tablename
    FROM pg_catalog.pg_tables
    WHERE schemaname = 'public'
    ORDER BY tablename;
""").df()

print(tablas)

   schemaname                              tablename
0      public                               Procesos
1      public               conteo_pasajeros_destino
2      public                            dias_semana
3      public                       estacionalidades
4      public                  estacionalidades_anio
5      public                      geography_columns
6      public                       geometry_columns
7      public                       novedades_viales
8      public                 novedades_viales_fecha
9      public                  novedades_viales_ruta
10     public                          progvehiculos
11     public                         raster_columns
12     public                       raster_overviews
13     public                                  rutas
14     public                        rutas_segmentos
15     public                 rutas_segmentos_config
16     public                rutas_segmentos_porcion
17     public                        spatial_r

In [7]:
df = con.execute("""
    SELECT r.idruta,
           ru.nombre                  AS nombre_ruta,
           COUNT(*)                   AS n_registros,
           COUNT(DISTINCT t.fecha)    AS n_dias,
           COUNT(DISTINCT t.id_viaje) AS n_viajes,
           MIN(t.fecha)               AS desde,
           MAX(t.fecha)               AS hasta
    FROM mi_pg.public.timbradas_segmentos t
    JOIN mi_pg.public.rutas_segmentos r  ON r.id = t.id_segmento
    JOIN mi_pg.public.rutas          ru ON ru.idruta = r.idruta
    WHERE r.idruta IN (16, 21, 2)
    GROUP BY r.idruta, ru.nombre
    ORDER BY r.idruta;
""").df()

print(df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   idruta nombre_ruta  n_registros  n_dias  n_viajes      desde      hasta
0       2    SILENCIO       799030      56      7519 2023-01-01 2023-02-27
1      16      CRA 54      1502215      56     14857 2023-01-01 2023-02-27
2      21       CL 72       494362      56      4649 2023-01-01 2023-02-27
